# 02 - Laboratuvar Temelli NumPy Modeli

Bu notebook, laboratuvarda gosterilen tek gizli katmanli MLP mantigini kalp yetmezligi veri setine uyarlayarak sifirdan egitir.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.data_utils import RANDOM_SEED, prepare_datasets, set_global_seed
from src.metrics import assess_fit_from_history, evaluate_classification
from src.numpy_models import NumpyMLPClassifier
from src.visualization import plot_confusion_matrix_figure, plot_training_history


In [ ]:
set_global_seed(RANDOM_SEED)
DATA_PATH = PROJECT_ROOT / 'data' / 'heart_failure_clinical_records_dataset.csv'
FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'
split = prepare_datasets(DATA_PATH, random_state=RANDOM_SEED)

baseline_model = NumpyMLPClassifier(
    layer_dims=[split.X_train.shape[1], 8, 1],
    learning_rate=0.05,
    epochs=400,
    seed=RANDOM_SEED,
    initialization='lab',
)
history = baseline_model.fit(split.X_train, split.y_train, split.X_val, split.y_val)

In [ ]:
train_predictions = baseline_model.predict(split.X_train)
validation_predictions = baseline_model.predict(split.X_val)
test_predictions = baseline_model.predict(split.X_test)

train_metrics = evaluate_classification(split.y_train, train_predictions, baseline_model.predict_proba(split.X_train))
validation_metrics = evaluate_classification(split.y_val, validation_predictions, baseline_model.predict_proba(split.X_val))
test_metrics = evaluate_classification(split.y_test, test_predictions, baseline_model.predict_proba(split.X_test))

pd.DataFrame([
    {'split': 'train', 'accuracy': train_metrics['accuracy'], 'precision': train_metrics['precision'], 'recall': train_metrics['recall'], 'specificity': train_metrics['specificity'], 'f1_score': train_metrics['f1_score']},
    {'split': 'validation', 'accuracy': validation_metrics['accuracy'], 'precision': validation_metrics['precision'], 'recall': validation_metrics['recall'], 'specificity': validation_metrics['specificity'], 'f1_score': validation_metrics['f1_score']},
    {'split': 'test', 'accuracy': test_metrics['accuracy'], 'precision': test_metrics['precision'], 'recall': test_metrics['recall'], 'specificity': test_metrics['specificity'], 'f1_score': test_metrics['f1_score']},
])

In [ ]:
plot_training_history(history, FIGURE_DIR / 'numpy_baseline', 'NumPy Baseline')
plot_confusion_matrix_figure(split.y_test, test_predictions, FIGURE_DIR / 'numpy_baseline_confusion_matrix.png', 'NumPy Baseline Confusion Matrix')

In [ ]:
print('Confusion matrix:', test_metrics['confusion_matrix'])
print('\nClassification report:\n')
print(test_metrics['classification_report'])

Yorum: Temel model, laboratuvarda gosterilen tek gizli katmanli yapinin kalp yetmezligi veri setine dogrudan uyarlanmis halidir. Agirliklar kucuk Gauss dagilimi ile, bias degerleri sifir ile baslatildigi icin parameter initialization mantigi da korunmustur.

In [ ]:
print(assess_fit_from_history(history))